# FloraGuard image model training

This notebook orchestrates the repository's scripts; it does not duplicate model or dataset logic and contains no generated/sample images. Add genuine, correctly labeled images and provenance rows before running. The source manifest must have one row per photo. Review `DATASET.md` first.

The split remains a group-aware manifest rather than copied image folders. Training evaluates only after selecting the model using validation data. Fine-tuning is optional and its checkpoint is retained only when validation loss improves.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

candidates = (Path.cwd(), *Path.cwd().parents)
ROOT = next((path for path in candidates if (path / 'train_model.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from a checkout containing train_model.py.')
DATA_DIR = ROOT / 'data' / 'rose'
SOURCE_DIR = DATA_DIR / 'source'
SOURCE_MANIFEST = DATA_DIR / 'source_manifest.csv'
MODEL_DIR = ROOT / 'models'
DATASET_VERSION = 'rose-data-1'
MODEL_VERSION = 'v1'
SEED = 42
BATCH_SIZE = 32
EPOCHS = 30
FINE_TUNE_EPOCHS = 0  # Set >0 to try validation-gated fine-tuning.
CONFIDENCE_THRESHOLD = 0.60

if any(importlib.util.find_spec(module) is None for module in ('tensorflow', 'numpy', 'PIL')):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt')])
print('Project root:', ROOT)
print('Data root:', DATA_DIR)

## 1. Inspect available hardware

An empty GPU list is normal. TensorFlow will use the CPU; this notebook does not install CUDA or drivers.

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('Visible GPUs:', tf.config.list_physical_devices('GPU'))

## 2. Validate real source images

This writes a review report without changing or deleting images. If no real images are present, stop here; do not make a split or train.

In [ ]:
subprocess.run([
    sys.executable, str(ROOT / 'dataset_tools.py'), 'validate',
    '--data-dir', str(SOURCE_DIR),
    '--output-dir', str(ROOT / 'reports' / 'dataset'),
], cwd=ROOT, check=True)
import json
validation = json.loads((ROOT / 'reports' / 'dataset' / 'dataset_validation.json').read_text(encoding='utf-8'))
print('Images found:', validation['valid_images'])
print('Images per class:', validation['counts_per_class'])
if validation['valid_images'] == 0:
    raise RuntimeError('No real source images found. Populate source class folders and source_manifest.csv before continuing.')

## 3. Create and inspect the reproducible split

In [ ]:
SPLIT_MANIFEST = DATA_DIR / 'split_manifest.csv'
subprocess.run([
    sys.executable, str(ROOT / 'dataset_tools.py'), 'split',
    '--data-dir', str(DATA_DIR),
    '--source-dir', str(SOURCE_DIR),
    '--source-manifest', str(SOURCE_MANIFEST),
    '--output', str(SPLIT_MANIFEST),
    '--seed', str(SEED),
], cwd=ROOT, check=True)

## 4. Train and evaluate

Training uses MobileNetV2 transfer learning. The script applies early stopping and evaluates the held-out test split after selecting on validation loss. Each run must use a new model version.

In [ ]:
command = [
    sys.executable, str(ROOT / 'train_model.py'),
    '--data-dir', str(DATA_DIR), '--output-dir', str(MODEL_DIR),
    '--manifest', str(SPLIT_MANIFEST), '--version', MODEL_VERSION,
    '--dataset-version', DATASET_VERSION, '--seed', str(SEED),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
    '--confidence-threshold', str(CONFIDENCE_THRESHOLD),
]
if FINE_TUNE_EPOCHS > 0:
    command.extend(['--fine-tune-epochs', str(FINE_TUNE_EPOCHS)])
subprocess.run(command, cwd=ROOT, check=True)

## 5. Review evaluation and confusion matrix

In [ ]:
from IPython.display import Image as NotebookImage, display

evaluation_path = MODEL_DIR / f'floraguard_{MODEL_VERSION}_evaluation.json'
evaluation = json.loads(evaluation_path.read_text(encoding='utf-8'))
print('Accuracy:', evaluation['accuracy'])
print('Macro average:', evaluation['macro_average'])
print('Per-class metrics:')
for class_name, metrics in evaluation['per_class'].items():
    print(class_name, metrics)
display(NotebookImage(filename=str(MODEL_DIR / f'floraguard_{MODEL_VERSION}_confusion_matrix.png')))
print('Review errors:', MODEL_DIR / f'floraguard_{MODEL_VERSION}_incorrect_predictions.csv')

## 6. Activate and smoke-test the held-out test image

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'activate_model.py'), '--model', str(MODEL_DIR / f'floraguard_{MODEL_VERSION}.keras')], cwd=ROOT, check=True)
subprocess.run([sys.executable, str(ROOT / 'test_predictor.py'), '--data-dir', str(DATA_DIR)], cwd=ROOT, check=True)

## 7. Export from Google Colab (optional)

Run this cell only in Colab after successful training. Save output to persistent storage if needed; Colab's temporary runtime filesystem is not persistent.

In [ ]:
if 'google.colab' in sys.modules:
    from google.colab import files
    for artifact in [
        MODEL_DIR / f'floraguard_{MODEL_VERSION}.keras',
        MODEL_DIR / f'floraguard_{MODEL_VERSION}_metadata.json',
        MODEL_DIR / f'floraguard_{MODEL_VERSION}_evaluation.json',
    ]:
        files.download(str(artifact))
else:
    print('This export cell is for Google Colab; artifacts are already in', MODEL_DIR)